# Read former files (``/mnt/hdd2/data/data_xrn2p/processed/matrix_solve``)

In [1]:
dir_name = "/mnt/hdd2/data/data_xrn2p/processed/matrix_solve"

In [2]:
import os
import polars as pl

In [26]:
import numpy as np


def filter_cv(df: pl.DataFrame, threshold: float=0.1, exclude_columns: list[str]=["Gene_ID"]):
    # Calculate the coefficient of variation (CV)
    df = df.fill_null(0.0)
    df = df.fill_nan(0.0)
    np_df_exclude = df.select(pl.selectors.float()).to_numpy()
    
    mean_values = np.mean(np_df_exclude, axis=1)
    cv = np.where(mean_values != 0, np.std(np_df_exclude, axis=1) / mean_values, 0)

    # Filter out features where CV < threshold
    filtered_df = df.filter(cv >= threshold)

    return filtered_df

## Read exp.matrix.tsv

这个大矩阵是由多个来源的数据拼接而成的，详情请见： ``/mnt/hdd2/data/data_xrn2p/processed/matrix_solve/rnaseq_matrix.ipynb``

In [8]:
exp_mat = pl.read_csv(os.path.join(dir_name, "exp.matrix.tsv.gz"), separator=",", has_header=True)
print(exp_mat.shape)

(29826, 1094)


In [9]:
exp_mat_f = filter_cv(exp_mat)
print(exp_mat_f)

shape: (16_633, 1_094)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Gene_ID   ┆ TPM_SRX46 ┆ TPM_SRX46 ┆ TPM_SRX46 ┆ … ┆ TPM_SRR19 ┆ TPM_SRR19 ┆ TPM_CRR15 ┆ TPM_CRR1 │
│ ---       ┆ 5187      ┆ 5188      ┆ 5189      ┆   ┆ 688219    ┆ 688222    ┆ 1235      ┆ 51236    │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ AT1G01010 ┆ 7.564866  ┆ 0.0       ┆ 2.551706  ┆ … ┆ 0.442365  ┆ 0.402581  ┆ 9.155551  ┆ 8.342136 │
│ AT1G01020 ┆ 19.834038 ┆ 5.972422  ┆ 7.47688   ┆ … ┆ 8.811214  ┆ 6.998165  ┆ 13.364632 ┆ 14.86934 │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ 5        │
│ AT1G01030 ┆ 3.087381  ┆ 1.688928  ┆ 2.109916  ┆ … ┆ 0.206472  ┆ 0.

### Write to parquet file

In [6]:
# exp_mat.write_parquet(os.path.join(dir_name, "exp.matrix.parquet"))

In [7]:
exp_mat_f.write_parquet(os.path.join(dir_name, "exp.matrix_cv0.1_.parquet"))

## Read ATAC

### Read atac_matrix.tsv

In [29]:
atac_mat = pl.read_csv(os.path.join(dir_name, "atac_matrix.tsv.gz"))
print(atac_mat.shape)
print(atac_mat.columns)

atac_mat = atac_mat.select(pl.col("geneId")).hstack(atac_mat.select(pl.selectors.float()))

# Drop a column if all values are null
atac_mat = atac_mat[[s.name for s in atac_mat if not (s.null_count() == atac_mat.height)]]
print(atac_mat)

(31394, 88)
['geneId', 'ath2017Unc1', 'ath2021Lc1', 'ath2018Cbb1', 'athPc2018Rbd4', 'athNp2019Rjs', 'athPc2018Rbd1', 'athPc2018Rbd3', 'ath2018Cbb2', 'athPc2018Rbd2', 'athPnasusa2021Sej20', 'ath2019Cnu2', 'ath2021Cos3', 'ath2021Ayu3', 'athSci2018Cjh2', 'athE2021Fb3', 'athPnasusa2021Sej19', 'athMbe2021Aey16', 'ath2021Ayu2', 'athMbe2021Aey6', 'athPnasusa2021Sej15', 'athPnasusa2021Sej1', 'athPnasusa2021Sej2', 'athPnasusa2021Sej18', 'athMbe2021Aey17', 'ath2017Unc5', 'athPj2018Rbd2', 'athMbe2021Aey7', 'athMbe2021Aey21', 'ath2020Caos2', 'athPnasusa2021Sej10', 'athMbe2021Aey14', 'athNc2021Yz1', 'athSci2018Cjh1', 'athMbe2021Aey3', 'athMbe2021Aey13', 'athJeb2019Ww1', 'athPnasusa2021Sej9', 'athMbe2021Aey8', 'athMbe2021Aey23', 'athPnasusa2021Sej11', 'athPnasusa2021Sej3', 'athNc2021Om1', 'athMbe2021Aey9', 'athNc2021Om3', 'ath2019Cnu1', 'athE2021Fb1', 'athMbe2021Aey1', 'athPnasusa2021Sej7', 'athMbe2021Aey10', 'athPnasusa2021Sej14', 'athPnasusa2021Sej4', 'athNc2021Yz3', 'ath2021Ayu1', 'athNc2021Yz6',

In [32]:
atac_mat_f = filter_cv(atac_mat, 0.1, ["geneId"])
print(atac_mat_f)

shape: (31_384, 86)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ geneId    ┆ ath2017Un ┆ ath2021Lc ┆ athPc2018 ┆ … ┆ athPnasus ┆ ath2019Cn ┆ athNc2021 ┆ athMbe20 │
│ ---       ┆ c1        ┆ 1         ┆ Rbd4      ┆   ┆ a2021Sej1 ┆ u4        ┆ Yz2       ┆ 21Aey15  │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ 2         ┆ ---       ┆ ---       ┆ ---      │
│           ┆ f64       ┆ f64       ┆ f64       ┆   ┆ ---       ┆ f64       ┆ f64       ┆ f64      │
│           ┆           ┆           ┆           ┆   ┆ f64       ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ AT1G01010 ┆ 17.72625  ┆ 19.525    ┆ 0.0       ┆ … ┆ 20.975    ┆ 21.475    ┆ 22.75     ┆ 17.7875  │
│ AT1G01020 ┆ 16.3925   ┆ 38.435    ┆ 0.0       ┆ … ┆ 22.76     ┆ 20.22     ┆ 0.0       ┆ 20.7875  │
│ AT1G01040 ┆ 27.505    ┆ 45.93     ┆ 0.0       ┆ … ┆ 24.91     ┆ 52.14

/tmp/ipykernel_632716/2836059241.py:11: RuntimeWarning: invalid value encountered in divide
  cv = np.where(mean_values != 0, np.std(np_df_exclude, axis=1) / mean_values, 0)


#### Write to parquet file

In [11]:
atac_mat.write_parquet(os.path.join(dir_name, "atac_matrix.parquet"))

### Read atac_matrix_1215.csv

In [7]:
atac_mat_1215 = pl.read_csv(os.path.join(dir_name, "atac_matrix_1215.csv"), schema_overrides={"Chr": pl.Utf8})
print(atac_mat_1215)
print(atac_mat_1215.columns)

shape: (1_777_262, 245)
┌─────┬────────┬──────────┬───────────┬───┬──────────────┬─────────────┬─────────────┬─────────────┐
│ Chr ┆ Start  ┆ End      ┆ geneId    ┆ … ┆ athMbe2021Ae ┆ athMbe2021A ┆ athMbe2021A ┆ athMbe2021A │
│ --- ┆ ---    ┆ ---      ┆ ---       ┆   ┆ y15IP1_RPKM  ┆ ey15IP2_RPK ┆ ey15IP3_RPK ┆ ey15IP4_RPK │
│ str ┆ i64    ┆ f64      ┆ str       ┆   ┆ ---          ┆ M           ┆ M           ┆ M           │
│     ┆        ┆          ┆           ┆   ┆ str          ┆ ---         ┆ ---         ┆ ---         │
│     ┆        ┆          ┆           ┆   ┆              ┆ str         ┆ str         ┆ str         │
╞═════╪════════╪══════════╪═══════════╪═══╪══════════════╪═════════════╪═════════════╪═════════════╡
│ 1   ┆ 1452   ┆ 1726.0   ┆ AT1G01010 ┆ … ┆ null         ┆ null        ┆ null        ┆ null        │
│ 1   ┆ 2384   ┆ 3752.0   ┆ AT1G01010 ┆ … ┆ null         ┆ null        ┆ null        ┆ null        │
│ 1   ┆ 8578   ┆ 9009.0   ┆ AT1G01020 ┆ … ┆ null         ┆ null    

#### Write to parquet file

In [9]:
atac_mat_1215.write_parquet(os.path.join(dir_name, "atac_matrix_1215.parquet"))

## Read m6A matrix

### Read m6a.matrix.tsv

In [11]:
m6a_mat = pl.read_csv(os.path.join(dir_name, "m6a.matrix.tsv.gz"))
print(m6a_mat.shape)
print(m6a_mat.columns)

(18874, 18)
['GeneID', 'data_26_Hk14v0x8Lu', 'Col-0_3h_flower', 'alkbh10b_3h_flower', 'Col-0_0h_leaf', 'Col-0_0h_flower', 'alkbh10b_3h_leaf', 'alkbh10b_0h_leaf', 'alkbh_10b_0h_flower', 'Col-0_3h_leaf', 'seqvir_4h', 'Col-0_4h', 'Col-0_0h', 'seqvir_0h', 'AmiR-mta_control', 'Col-0_control', 'Col-0_chilling', 'AmiR-mta_chilling']


In [12]:
# m6a_mat.write_parquet(os.path.join(dir_name, "m6a.matrix.parquet"))

m6a_mat_f = filter_cv(m6a_mat, exclude_columns=["GeneID"])
print(m6a_mat_f)
m6a_mat_f.write_parquet(os.path.join(dir_name, "m6a.matrix_cv0.1_.parquet"))

shape: (373, 18)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ GeneID    ┆ data_26_H ┆ Col-0_3h_ ┆ alkbh10b_ ┆ … ┆ AmiR-mta_ ┆ Col-0_con ┆ Col-0_chi ┆ AmiR-mta │
│ ---       ┆ k14v0x8Lu ┆ flower    ┆ 3h_flower ┆   ┆ control   ┆ trol      ┆ lling     ┆ _chillin │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ g        │
│           ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ AT1G01160 ┆ 6.020132  ┆ 6.236381  ┆ 5.67623   ┆ … ┆ 2.086673  ┆ 3.584082  ┆ 2.62113   ┆ 1.991074 │
│ AT1G01350 ┆ 9.974834  ┆ 8.505584  ┆ 6.819328  ┆ … ┆ 2.171572  ┆ 3.054322  ┆ 2.532037  ┆ 2.170916 │
│ AT1G02100 ┆ 28.362232 ┆ 8.548039  ┆ 7.862305  ┆ … ┆ 3.042872  ┆ 5.990745

/home/wuch/miniforge3/envs/pyplot/lib/python3.12/site-packages/numpy/_core/_methods.py:185: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


### Read m6A_matrix2.tsv

In [5]:
m6a_mat2 = pl.read_csv(os.path.join(dir_name, "m6a_matrix2.tsv"), separator="\t", null_values="NA")
print(m6a_mat2)
print(m6a_mat2.columns)

shape: (330_336, 82)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ geneId    ┆ alkbh_10b ┆ alkbh_10b ┆ Col.0_chi ┆ … ┆ athMbe202 ┆ athMbe202 ┆ athGb2015 ┆ athGb201 │
│ ---       ┆ _0h_flowe ┆ _0h_flowe ┆ lling.rep ┆   ┆ 1Mc.rep2  ┆ 1Mc.rep3  ┆ Zbl3.rep1 ┆ 5Zbl3.re │
│ str       ┆ r.rep1    ┆ r.rep2    ┆ 1         ┆   ┆ ---       ┆ ---       ┆ ---       ┆ p2       │
│           ┆ ---       ┆ ---       ┆ ---       ┆   ┆ str       ┆ str       ┆ f64       ┆ ---      │
│           ┆ str       ┆ f64       ┆ f64       ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ AT1G03997 ┆ 3.2553803 ┆ 2.598784  ┆ null      ┆ … ┆ null      ┆ null      ┆ null      ┆ null     │
│           ┆ 6610455   ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ AT1G04013 ┆ 4.9029699 ┆ 3.263834  ┆ null      ┆ … ┆ null      ┆ null

In [ ]:
m6a_mat2.write_parquet(os.path.join(dir_name, "m6a_matrix2.parquet"))

## Read prot

In [9]:
prot_mat = pl.read_csv(os.path.join(dir_name, "prot_matrix.csv.gz"))
print(prot_mat)
print(prot_mat.columns)

shape: (10_195, 191)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Protein_I ┆ PXD019537 ┆ PXD019537 ┆ PXD019537 ┆ … ┆ PXD040584 ┆ PXD040584 ┆ PXD040584 ┆ PXD04058 │
│ Ds        ┆ _LFQ_inte ┆ _LFQ_inte ┆ _LFQ_inte ┆   ┆ _LFQ_inte ┆ _LFQ_inte ┆ _LFQ_inte ┆ 4_LFQ_in │
│ ---       ┆ nsity_123 ┆ nsity_123 ┆ nsity_123 ┆   ┆ nsity_Cyt ┆ nsity_Cyt ┆ nsity_Cyt ┆ tensity_ │
│ str       ┆ 87-…      ┆ 86-…      ┆ 85-…      ┆   ┆ o_C…      ┆ o_C…      ┆ o_C…      ┆ Cyto_C…  │
│           ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ AT1G67120 ┆ 9.5629e7  ┆ 0.0       ┆ 0.0       ┆ … ┆ 9.7761e6  ┆ 2.4302e6  ┆ 1.1008e7  ┆ 1.107e7  │
│ AT5G38890 ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0 

In [ ]:
prot_mat.write_parquet(os.path.join(dir_name, "prot_matrix.parquet"))

### Read data_06_21gmagS9Ul data_06_ibaq_phos_rnaseq.csv

In [5]:
prot_phos_rna_mat = pl.read_csv("/mnt/hdd2/data/data_xrn2p/processed/data_06_21gmagS9Ul/data_06_ibaq_phos_rnaseq.csv")
print(prot_phos_rna_mat)
print(prot_phos_rna_mat.columns)
print(prot_phos_rna_mat.columns.__len__() - 1)

shape: (35_102, 30)
┌────────────────┬─────────┬──────────┬──────────┬───┬──────────┬──────────┬──────────┬──────────┐
│ Gene_ID        ┆ SP      ┆ PT       ┆ ST       ┆ … ┆ HY       ┆ RTTP     ┆ RTUZ     ┆ CTSAM    │
│ ---            ┆ ---     ┆ ---      ┆ ---      ┆   ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ str            ┆ f64     ┆ f64      ┆ f64      ┆   ┆ f64      ┆ f64      ┆ f64      ┆ f64      │
╞════════════════╪═════════╪══════════╪══════════╪═══╪══════════╪══════════╪══════════╪══════════╡
│ iBAQ_AT1G01080 ┆ 27.4709 ┆ 24.6839  ┆ 23.9193  ┆ … ┆ 26.9522  ┆ 19.6416  ┆ 21.7506  ┆ 28.1953  │
│ iBAQ_AT1G01090 ┆ 26.8864 ┆ 29.1833  ┆ 28.7721  ┆ … ┆ 28.7368  ┆ 29.0178  ┆ 28.7675  ┆ 29.1401  │
│ iBAQ_AT1G01100 ┆ 26.3901 ┆ 26.2492  ┆ 26.001   ┆ … ┆ 26.3261  ┆ 26.8466  ┆ 25.8308  ┆ 26.1496  │
│ iBAQ_AT1G01220 ┆ 23.7557 ┆ 23.7033  ┆ 23.8349  ┆ … ┆ 24.3392  ┆ 23.4844  ┆ 22.8559  ┆ 23.6357  │
│ iBAQ_AT1G01260 ┆ 20.2059 ┆ 21.4494  ┆ 19.8481  ┆ … ┆ 20.5721  ┆ 22.643   ┆ 22.6052  ┆ 2

In [6]:
prot_phos_rna_mat.write_parquet("/mnt/hdd2/homext/wuch/xn2p/data/raw_df/data_06_prot_phos_rna.parquet")

#### 检查第一列的列名有多少类型

In [8]:
import numpy as np

In [9]:
feature_ids = prot_phos_rna_mat.select("Gene_ID").to_series().to_list()
feature_types = [i.split("_")[0] for i in feature_ids]
omics_types = np.unique_counts(feature_types)
print(omics_types)

UniqueCountsResult(values=array(['IntensityP', 'TPM', 'iBAQ'], dtype='<U10'), counts=array([ 1656, 25712,  7734]))
